In [1]:
import pandas as pd
import re
import numpy as np
from scipy.stats import mannwhitneyu
import krippendorff
from collections import Counter
from scipy.stats import wilcoxon

In [3]:
# Paths
df_combined_all_groups = {}
for group in ["biology", "gamedev", "philosophy"]:
    print(group)
    input_path = f"human_study_results_{group}.csv"
    output_path = f"human_study_qwen_{group}.csv"

    # Load CSV (single row)
    df = pd.read_csv(input_path)

    # Likert-scale mapping
    likert_map = {
        "Not at all": 1,
        "Rather not": 2,
        "Partly, partly not": 3,
        "Rather": 4,
        "Fully": 5
    }

    exp_pattern = re.compile(r"\[Explanation ([ABC])\]")

    results = {}
    for row_index, row in df.iterrows():
        results[row_index] = {"Q1A": [], "Q1B": [], "Q1C": [], "Q2A": [], "Q2B": [], "Q2C": [], "Q3A": [], "Q3B": [], "Q3C": []}
        #print(row)
        i = 1
        count = 0
        for series_name, series in row.items():
            match = exp_pattern.search(series_name)
            if not match:
                continue
            exp = match.group(1)
            if count > 2:
                count = 0
                i += 1
            if i == 4:
                i = 1
            results[row_index][f"Q{i}{exp}"].append(likert_map[series])
            count += 1

    merged_results = {"Q1A": [], "Q1B": [], "Q1C": [], "Q2A": [], "Q2B": [], "Q2C": [], "Q3A": [], "Q3B": [], "Q3C": []}
    for criteria in merged_results:
        all_labels = []
        for label1, label2, label3 in zip(results[0][criteria], results[1][criteria], results[2][criteria]):
            merged_results[criteria].append([label1,label2,label3])
    
    merged_results_df = pd.DataFrame(merged_results)
    #print(merged_results_df)

    data = pd.read_csv(output_path)
    #print(data)

    desired_order = ["prompt_group", "activation_konen_group", "activation_group_attribute_5"]
    # Create a mapping from element to desired index
    order_map = {key: i for i, key in enumerate(desired_order)}
    
    df_combined = pd.concat([data, merged_results_df], axis=1)

    for dataset_name, df_combined in zip(["eli5", "rooein", "both"], [df_combined[25:], df_combined[:25], df_combined]):
        print(dataset_name)
        if dataset_name not in df_combined_all_groups:
            df_combined_all_groups[dataset_name] = []
        for row_index, row in df_combined.iterrows():
            approaches = row[["Approach A", "Approach B", "Approach C"]].values
            #print(approaches.values)
            sorted_indices = np.argsort([order_map[x] for x in approaches])

            approaches_sorted = approaches[sorted_indices]
           
            explanations_sorted = row[["Explanation A", "Explanation B", "Explanation C"]].values[sorted_indices]
            
            criteria_1_sorted = row[["Q1A", "Q1B", "Q1C"]].values[sorted_indices]
            criteria_2_sorted = row[["Q2A", "Q2B", "Q2C"]].values[sorted_indices]
            criteria_3_sorted = row[["Q3A", "Q3B", "Q3C"]].values[sorted_indices]

            df_combined.loc[row_index, ["Approach A", "Approach B", "Approach C"]] = approaches_sorted
            df_combined.loc[row_index, ["Explanation A", "Explanation B", "Explanation C"]] = explanations_sorted
            df_combined.loc[row_index, ["Q1A", "Q1B", "Q1C"]] = criteria_1_sorted
            df_combined.loc[row_index, ["Q2A", "Q2B", "Q2C"]] = criteria_2_sorted
            df_combined.loc[row_index, ["Q3A", "Q3B", "Q3C"]] = criteria_3_sorted
        
        if len(df_combined_all_groups[dataset_name]) == 0:
            df_combined_all_groups[dataset_name] = df_combined[["Q1A", "Q1B", "Q1C", "Q2A", "Q2B", "Q2C", "Q3A", "Q3B", "Q3C"]]
        else:    
            df_combined_all_groups[dataset_name] = df_combined_all_groups[dataset_name].combine(df_combined[["Q1A", "Q1B", "Q1C", "Q2A", "Q2B", "Q2C", "Q3A", "Q3B", "Q3C"]], lambda x1, x2: x1 + x2)
        
        means = {}
        for row_index, row in df_combined.iterrows():
            for approach in ["A", "B", "C"]:
                if approach not in means:
                    means[approach] = {}
                for criterion in ["Q1", "Q2", "Q3"]:
                    if criterion not in means[approach]:
                            means[approach][criterion] = []
                    means[approach][criterion].append(row[f"{criterion}{approach}"])
        
        def majority_vote(labels):
            counts = Counter(labels)
            label, count = counts.most_common(1)[0]
            return label if count >= 2 else np.mean(labels)

        for approach, approach_name in zip(["A", "B", "C"], desired_order):
            for criterion in ["Q1", "Q2", "Q3"]:
                
                # Mean
                mean = np.mean(means[approach][criterion])
                
                # Majority
                majorities = []
                for labels in means[approach][criterion]:
                    majorities.append(majority_vote(labels))
                majority = np.mean(majorities) 
                print(approach_name, criterion, mean, majority)
                
                if group == "gamedev":
                    alpha = krippendorff.alpha(reliability_data=np.array(means[approach][criterion]).T, value_domain=[1, 2, 3, 4, 5], level_of_measurement="ordinal")
                    #print("Krippendorff alpha:", alpha)
        for criterion in ["Q1", "Q2", "Q3"]:
            if criterion == "Q1":
                stat, p = wilcoxon(
                    np.array(means["C"][criterion]).flatten(),
                    np.array(means["A"][criterion]).flatten(), 
                    alternative="greater"
                )

                print(criterion, "Is activation_group_attribute_5 better than prompt_group?")
                #print("Stat:", stat)
                print("p-value:", p)

                stat, p = wilcoxon(
                    np.array(means["C"][criterion]).flatten(),
                    np.array(means["B"][criterion]).flatten(), 
                    alternative="greater"
                )

                print(criterion, "Is activation_group_attribute_5 better than activation_konen_group?")
                #print("Stat:", stat)
                print("p-value:", p)
            else:
                stat, p = wilcoxon(
                    np.array(means["C"][criterion]).flatten(),
                    np.array(means["A"][criterion]).flatten(), 
                    alternative="greater"
                )

                print(criterion, "Is activation_group_attribute_5 better than prompt_group?")
                #print("Stat:", stat)
                print("p-value:", p)
                
                stat, p = wilcoxon(
                    np.array(means["C"][criterion]).flatten(),
                    np.array(means["A"][criterion]).flatten(), 
                    alternative="less"
                )

                print(criterion, "Is activation_group_attribute_5 worse than prompt_group?")
                #print("Stat:", stat)
                print("p-value:", p)

                stat, p = wilcoxon(
                    np.array(means["C"][criterion]).flatten(),
                    np.array(means["B"][criterion]).flatten(), 
                    alternative="greater"
                )

                print(criterion, "Is activation_group_attribute_5 better than activation_konen_group?")
                #print("Stat:", stat)
                print("p-value:", p)

        print("---------------------------------------------------------")

def harmonic_means_three(criterion1, criterion2, criterion3):
    hmeans = []
    for x1, x2, x3 in zip(criterion1, criterion2, criterion3):
        numerator = 3 * x1 * x2 * x3
        denominator = (x1*x2 + x1*x3 + x2*x3)
        hmeans.append(numerator / denominator)
    return hmeans

def harmonic_mean_three(x1, x2, x3):
    numerator = 3 * x1 * x2 * x3
    denominator = (x1*x2 + x1*x3 + x2*x3)
    hmean = numerator / denominator
    return hmean
        
print("All groups")
for dataset_name in df_combined_all_groups:
    print(dataset_name)
    df_combined = df_combined_all_groups[dataset_name]
    #print(df_combined)
    means = {}
    hmeans = {}
    for row_index, row in df_combined.iterrows():
        for approach in ["A", "B", "C"]:
            if approach not in means:
                means[approach] = {}
                hmeans[approach] = []
            for criterion in ["Q1", "Q2", "Q3"]:
                if criterion not in means[approach]:
                    means[approach][criterion] = []
                means[approach][criterion].append(row[f"{criterion}{approach}"])
            hmeans[approach].append(harmonic_means_three(row[f"Q1{approach}"], row[f"Q2{approach}"], row[f"Q3{approach}"]))

    def majority_vote(labels):
        counts = Counter(labels)
        label, count = counts.most_common(1)[0]
        return label if count >= 5 else np.mean(labels)

    for approach, approach_name in zip(["A", "B", "C"], desired_order):
        for criterion in ["Q1", "Q2", "Q3"]:
            # Mean
            mean = np.mean(means[approach][criterion])
            std = np.std(means[approach][criterion])

            # Majority
            majorities = []
            for labels in means[approach][criterion]:
                majorities.append(majority_vote(labels))
            majority = np.mean(majorities) 
            #print(approach_name, criterion, mean, majority)
            alpha = krippendorff.alpha(reliability_data=np.array(means[approach][criterion]), value_domain=[1, 2, 3, 4, 5], level_of_measurement="ordinal")
            #print("Krippendorff alpha:", alpha)
            print(approach_name, criterion, mean, std, alpha)
        # Harmonic mean
        hmean = np.mean(hmeans[approach])
        print("Harmonic mean:", harmonic_mean_three(np.mean(means[approach]["Q1"]), np.mean(means[approach]["Q2"]), np.mean(means[approach]["Q3"])))
        hmean = np.mean(hmeans[approach])
        print("Harmonic mean (sample-wise):", hmean)
        
    # Significance difference based on each criterion
    for criterion in ["Q1", "Q2", "Q3"]:
        if criterion == "Q1":
            stat, p = wilcoxon(
                np.array(means["C"][criterion]).flatten(),
                np.array(means["A"][criterion]).flatten(), 
                alternative="greater"
            )
            
            print(np.array(means["C"][criterion]).flatten())
            print(np.array(means["A"][criterion]).flatten())

            print(criterion, "Is activation_group_attribute_5 better than prompt_group?")
            #print("Stat:", stat)
            print("p-value:", p)

            stat, p = wilcoxon(
                np.array(means["C"][criterion]).flatten(),
                np.array(means["B"][criterion]).flatten(), 
                alternative="greater"
            )

            print(criterion, "Is activation_group_attribute_5 better than activation_konen_group?")
            #print("Stat:", stat)
            print("p-value:", p)
        else:
            stat, p = wilcoxon(
                np.array(means["C"][criterion]).flatten(),
                np.array(means["A"][criterion]).flatten(), 
                alternative="greater"
            )

            print(criterion, "Is activation_group_attribute_5 better than prompt_group?")
            #print("Stat:", stat)
            print("p-value:", p)
            
            stat, p = wilcoxon(
                np.array(means["C"][criterion]).flatten(),
                np.array(means["A"][criterion]).flatten(), 
                alternative="less"
            )

            print(criterion, "Is activation_group_attribute_5 worse than prompt_group?")
            #print("Stat:", stat)
            print("p-value:", p)

            stat, p = wilcoxon(
                np.array(means["C"][criterion]).flatten(),
                np.array(means["B"][criterion]).flatten(), 
                alternative="greater"
            )

            print(criterion, "Is activation_group_attribute_5 better than activation_konen_group?")
            #print("Stat:", stat)
            print("p-value:", p)
    
    # Significance difference based on harmonic mean of all criteria
    stat, p = wilcoxon(
        np.array(hmeans["C"]).flatten(),
        np.array(hmeans["A"]).flatten(), 
        alternative="greater"
        
    )
    #print(np.array(hmeans["C"]).flatten())
    #print(np.array(hmeans["A"]).flatten())
    

    print("Hmean: Is activation_group_attribute_5 better than prompt_group?")
    #print("Stat:", stat)
    print("p-value:", p)

    stat, p = wilcoxon(
        np.array(hmeans["C"]).flatten(),
        np.array(hmeans["B"]).flatten(), 
        alternative="greater"
        
    )

    print("Hmean: Is activation_group_attribute_5 better than activation_konen_group?")
    #print("Stat:", stat)
    print("p-value:", p)
    

    print("---------------------------------------------------------")
    
    #df_combined_all_groups[dataset_name]
    
    #count = {"A": 0, "B": 0, "C": 0}
    #for row_index, row in df_combined_all_groups["both"].iterrows():
    #    for helpfulnessA, helpfulnessB, helpfulnessC in zip(row[f"Q2A"], row[f"Q2B"], row[f"Q2C"]):
    #        if helpfulnessA > helpfulnessB and helpfulnessA > helpfulnessC:
    #            count["A"] += 1
    #        elif helpfulnessB > helpfulnessA and helpfulnessB > helpfulnessC:
    #            count["B"] += 1
    #        elif helpfulnessC > helpfulnessA and helpfulnessC > helpfulnessB:
    #            count["C"] += 1
    #print(count)
    

biology
eli5
prompt_group Q1 3.2133333333333334 3.0666666666666664
prompt_group Q2 4.666666666666667 4.88
prompt_group Q3 4.76 4.92
activation_konen_group Q1 3.12 3.0533333333333332
activation_konen_group Q2 3.2933333333333334 3.24
activation_konen_group Q3 3.32 3.3466666666666662
activation_group_attribute_5 Q1 3.88 4.133333333333334
activation_group_attribute_5 Q2 3.4133333333333336 3.56
activation_group_attribute_5 Q3 3.466666666666667 3.6266666666666665
Q1 Is activation_group_attribute_5 better than prompt_group?
p-value: 0.0016508008315364896
Q1 Is activation_group_attribute_5 better than activation_konen_group?
p-value: 7.911526249326683e-05
Q2 Is activation_group_attribute_5 better than prompt_group?
p-value: 0.9999999602912867
Q2 Is activation_group_attribute_5 worse than prompt_group?
p-value: 3.9708713311931906e-08
Q2 Is activation_group_attribute_5 better than activation_konen_group?
p-value: 0.26620389929469346
Q3 Is activation_group_attribute_5 better than prompt_group?
p-